In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

In [2]:
# Define State
class LLMState(TypedDict):
    question: str
    answer: str

In [3]:
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash", temperature=0.2 
)

In [4]:
def llm_qa(state: LLMState) -> LLMState:
    # extract the question from state
    question = state['question']

    # form the prompt
    prompt = f"Answer the following question {question}. answer in simple words and use at max 2 lines no more then 2 lines."

    # ask the questoin to llm
    answer = llm.invoke(prompt).content

    # update the state
    state['answer'] = answer

    return state

In [5]:
graph = StateGraph(LLMState)

# define the nodes
graph.add_node('llm_qa', llm_qa)

# define edges
graph.add_edge(START, 'llm_qa')
graph.add_edge('llm_qa', END)

workflow = graph.compile()

In [6]:
initial_state = {
    "question": "Who is the prime minister of germany"
}

final_state = workflow.invoke(initial_state)

final_state

{'question': 'Who is the prime minister of germany',
 'answer': "Germany's head of government is called the Chancellor, not Prime Minister.\nThe current Chancellor of Germany is Olaf Scholz."}